# Lab 7: Contextual Bag of Words with Pytorch
### COSC 426: Fall 2025, Colgate University

Use this notebook to answer the questions in `Lab7.md`. Make sure to include in this notebook all the tests and experiments you run. Make sure to also cite any external resources you use. 

## Part 1: Familiarize yourself with the different components of the CBOW model

Add as many code chunks and markdown chunks as required to answer the questions in this part.

In [2]:
import CBOW
import torch

### Part 1.1


#### 1.

In [13]:
# create CBOW Dataset
sample_data = CBOW.CBOW_Dataset('data/sample-alice.txt', 'data/sample_vocab.txt', window_size=4)

# number of training examples
sample_data_size = len(sample_data)
print(f"Number of training examples: {sample_data_size}")

Number of training examples: 336


#### 2.

In [12]:
# create CBOW Dataset
sample_data_2 = CBOW.CBOW_Dataset('data/sample-alice.txt', 'data/sample_vocab.txt', window_size=2)

# number of training examples
sample_data_size_2 = len(sample_data_2)
print(f"Number of training examples: {sample_data_size_2}")

Number of training examples: 336


#### 3.

No. `make_pairs()` has a loop that make each token a target. With padding, no matter what `window_size` it is, the number of pairs is the number of tokens. 

#### 4. 

The raw input is just text. It will be tokenized and for each token in the input (`y`), the corresponding `X` will be the set of tokens before and after it with the length of `window_size`. It goes through tokenization, vocab mapping, encoding, then making pairs with padding, and append as tensor format.

In [15]:
print("Sample training example (window size = 4):")
context_vector, target_word = sample_data[0]
print("Context word indices:", context_vector)
print("Target word index:", target_word)

Sample training example (window size = 4):
Context word indices: tensor([  0,   0,   0,   0,  21,  59, 122, 109])
Target word index: tensor(106)


#### 5.

No. When loading the data, it's set to lower cased. To make it cases-sensitive, remove `.lower()`

### Part 1.2


#### 1.

The embedding layer converts high-dimensional data to a lower dimension. It's condensing the text information for the model to learn that more easily.

The input dimension is the same as vocab size. The output dimension is the targeted embedding dimension we set (`nEmbed`).

Embedding automatically use one-hot encoding so the orginal vector size is one by vocab size. Original token ID doesn't contain semantic information. After embedding, words with similar context will be closed in the space.

#### 2.

The linear layer takes the average context embedding the predicts which word (target) should be there.

The input dimension is the targeted embedding dimension we set (`nEmbed`). The output dimension is the vocab size.

We want know the one word out of the vocab. For each sample, we want the output to be a distribution over the vocab.

#### 3.

Convert context word IDs into embeddings. Take the average of all embeddings to get one combined context representation. Feed this average into a linear layer to produce scores for every possible target word.

#### 4.

For each sample, we take cross-entropy loss, which automatically deal with probabilities. The model learns to assign higher scores to the correct target words, shaping the embeddings to reflect patterns in the data.

### Part 1.3


#### 1.

* `num_epochs`: the number of times the trainer goes through the training data
* `lr`: learning-rating shows how much to change in reponse to the loss
* `batch_size` : the number of samples in each iteration (impacts how many times the weights get updated)
* `train_data`: dataset used for training to learn the weights
* `val_data`: dataset used for validation to confirm the weights capturing the patterns (printing the loss for the reference)
* `device`: where training will take place (CPU/GPU)

#### 2.

``` python
DataLoader(dataset, batch_size=1, shuffle=False, sampler=None,
           batch_sampler=None, num_workers=0, collate_fn=None,
           pin_memory=False, drop_last=False, timeout=0,
           worker_init_fn=None, *, prefetch_factor=2,
           persistent_workers=False)
```

We are passing 
* dataset from which to load the data
* batch size - how many samples per batch to load (default: 1)
* if the dataset should be reshuffled at every epoch (default: False)

The output is an iterable over the given dataset.

#### 3.

* `X,y_target = X.to(self.device), y_target.to(self.device)`: put training data to the device where the process takes place
* `loss = model.loss(y_pred, y_target)`: compute the loss of the prediction again the gold data
* `optimizer.zero_grad()`: clear the gradients from previous iterations
* `loss.backward()`: initiate the backpropagation to compute the gradients of the loss function with respect to the parameters
* `optimizer.step()`: update the parameters (weights) based on the computed gradients

### Part 1.4


#### 1.

* `compute_loss` loops over the entire test set. For each batch, run the model to get predictions, calculate the loss, averages the loss over all batches and return the average.
* `get_preds` loops over the entire test set as well. For each batch, compute `y_preds`, take the `argmax` across vocab size dimension to get the predicted label and also collect the gold label. This is closer to evaluate as it provides output for each sample.

#### 2.

`@torch.nograd` prevents track operations for gradient computation as we don't need gradients in these evaluation steps. It saves memory.

#### 3.

```python
evaluator = CBOW_Evaluator(test_data, batch_size=32, device='cpu')
test_loss = evaluator.compute_loss(model)
```

## Part 2: Train and evaluate a CBOW model on toy data, and explore the word embeddings

Add as many code chunks and markdown chunks as required to answer the questions in this part.


In [16]:
## Function to get device

def get_device(args):
    use_cuda = not args['cuda'] and torch.cuda.is_available()
    use_mps = not args['mps'] and torch.backends.mps.is_available()

    if use_cuda:
        device = torch.device("cuda")
    elif use_mps:
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    return device


### Initialize model and datasets

In [43]:
train_data = CBOW.CBOW_Dataset('data/sample-alice.txt', 'data/sample_vocab.txt', window_size=4)
val_data = CBOW.CBOW_Dataset('data/sample-lookingglass.txt', 'data/sample_vocab.txt', window_size=4)

toy_model = CBOW.CBOW_Model(nEmbed = 50, vocabSize=train_data.vocabSize)

### Evaluate randomly initialized model

* Report loss and accuracy on the training data
* Report cosine similarity between the 4 word pairs

In [44]:
from sklearn.metrics import accuracy_score 
evaluator = CBOW.CBOW_Evaluator(train_data, batch_size=32, device=torch.device("cpu"))

rand_loss = evaluator.compute_loss(toy_model)
print(f"Random model loss on training data: {rand_loss}")

rand_pred, rand_target = evaluator.get_preds(toy_model)
rand_accuracy = accuracy_score(rand_target, rand_pred)
print (f"Random model accuracy on training data: {rand_accuracy}")

Random model loss on training data: 5.074333451010964
Random model accuracy on training data: 0.005952380952380952


In [45]:
def cosine_similarity(vec1, vec2):
    dot_product = torch.dot(vec1, vec2)
    norm1 = torch.norm(vec1)
    norm2 = torch.norm(vec2)
    return dot_product / (norm1 * norm2)

def cosine_similarity_word(word1, word2, embedding_matrix, data):
    idx1 = data.word_to_id[word1]
    idx2 = data.word_to_id[word2]
    vec1 = embedding_matrix[idx1]
    vec2 = embedding_matrix[idx2]
    return cosine_similarity(vec1, vec2).item()

embedding_matrix = toy_model.embed.weight.data

word1 = "think"
word2 = "thought"
similarity = cosine_similarity_word(word1, word2, embedding_matrix, train_data)
print(f"Cosine similarity between '{word1}' and '{word2}': {similarity}")

word1 = "think"
word2 = "tired"
similarity = cosine_similarity_word(word1, word2, embedding_matrix, train_data)
print(f"Cosine similarity between '{word1}' and '{word2}': {similarity}")

word1 = "sleepy"
word2 = "thought"
similarity = cosine_similarity_word(word1, word2, embedding_matrix, train_data)
print(f"Cosine similarity between '{word1}' and '{word2}': {similarity}")

word1 = "sleepy"
word2 = "tired"
similarity = cosine_similarity_word(word1, word2, embedding_matrix, train_data)
print(f"Cosine similarity between '{word1}' and '{word2}': {similarity}")

Cosine similarity between 'think' and 'thought': -0.07208263128995895
Cosine similarity between 'think' and 'tired': 0.13996915519237518
Cosine similarity between 'sleepy' and 'thought': -0.06127794459462166
Cosine similarity between 'sleepy' and 'tired': -0.26646924018859863


### Train the model

In [47]:
trainer = CBOW.CBOW_Trainer(num_epochs = 1000, lr = 0.5, batch_size = 8, train_data = train_data, val_data = val_data, device = torch.device("cpu"))
trainer.train(toy_model)

Epoch 0:	 Avg Train Loss: 0.00734	 Avg Val Loss: 12.61055
Epoch 20:	 Avg Train Loss: 0.00684	 Avg Val Loss: 12.6307
Epoch 40:	 Avg Train Loss: 0.00919	 Avg Val Loss: 12.63332
Epoch 60:	 Avg Train Loss: 0.00819	 Avg Val Loss: 12.65306
Epoch 80:	 Avg Train Loss: 0.00799	 Avg Val Loss: 12.67932
Epoch 100:	 Avg Train Loss: 0.00702	 Avg Val Loss: 12.68778
Epoch 120:	 Avg Train Loss: 0.00707	 Avg Val Loss: 12.71505
Epoch 140:	 Avg Train Loss: 0.00793	 Avg Val Loss: 12.72154
Epoch 160:	 Avg Train Loss: 0.00923	 Avg Val Loss: 12.76087
Epoch 180:	 Avg Train Loss: 0.00548	 Avg Val Loss: 12.7691
Epoch 200:	 Avg Train Loss: 0.00636	 Avg Val Loss: 12.79917
Epoch 220:	 Avg Train Loss: 0.00637	 Avg Val Loss: 12.79019
Epoch 240:	 Avg Train Loss: 0.01463	 Avg Val Loss: 12.82846
Epoch 260:	 Avg Train Loss: 0.00827	 Avg Val Loss: 12.82157
Epoch 280:	 Avg Train Loss: 0.0073	 Avg Val Loss: 12.83947
Epoch 300:	 Avg Train Loss: 0.00824	 Avg Val Loss: 12.84531
Epoch 320:	 Avg Train Loss: 0.00667	 Avg Val Loss

### Evaluate trained model

* Report loss and accuracy on the training data
* Report cosine similarity between the 4 word pairs

In [48]:
from sklearn.metrics import accuracy_score 
evaluator = CBOW.CBOW_Evaluator(train_data, batch_size=32, device=torch.device("cpu"))

rand_loss = evaluator.compute_loss(toy_model)
print(f"Random model loss on training data: {rand_loss}")

rand_pred, rand_target = evaluator.get_preds(toy_model)
rand_accuracy = accuracy_score(rand_target, rand_pred)
print (f"Random model accuracy on training data: {rand_accuracy}")

Random model loss on training data: 0.00613254739172672
Random model accuracy on training data: 0.9970238095238095


In [49]:
embedding_matrix = toy_model.embed.weight.data

word1 = "think"
word2 = "thought"
similarity = cosine_similarity_word(word1, word2, embedding_matrix, train_data)
print(f"Cosine similarity between '{word1}' and '{word2}': {similarity}")

word1 = "think"
word2 = "tired"
similarity = cosine_similarity_word(word1, word2, embedding_matrix, train_data)
print(f"Cosine similarity between '{word1}' and '{word2}': {similarity}")

word1 = "sleepy"
word2 = "thought"
similarity = cosine_similarity_word(word1, word2, embedding_matrix, train_data)
print(f"Cosine similarity between '{word1}' and '{word2}': {similarity}")

word1 = "sleepy"
word2 = "tired"
similarity = cosine_similarity_word(word1, word2, embedding_matrix, train_data)
print(f"Cosine similarity between '{word1}' and '{word2}': {similarity}")

Cosine similarity between 'think' and 'thought': 0.40050336718559265
Cosine similarity between 'think' and 'tired': 0.04400194063782692
Cosine similarity between 'sleepy' and 'thought': 0.02348792552947998
Cosine similarity between 'sleepy' and 'tired': -0.24565327167510986


### Discussion and reflection

* Do you think that the model has learned the task? Do you think the model has learned useful embeddings?

* Does it make sense to use embedding size of 300 for this toy data? Why or why not? 

* Accuracy of 0.997 means it has effectively learned to predict target words from context. It partially learned embeddings as the similarity between “think” and “thought” improved, indicating the embeddings now encode some semantic structure.
However, other pairs suggest limited generalization.

* Using 300-dimensional embeddings for a dataset with only 147 vocabulary tokens is not a good idea. That’s large relative to the amount of information in the dataset and makes overfitting very likely. The model can memorize training examples without learning robust semantic structure. Also, higher dimensional embeddings increase training time and memory, with little to no benefit on small data.

## Part 3: Explore the role of training data on the word embeddings that are learned

### Train models

Answer question 1 here

In [3]:
alice_data = CBOW.CBOW_Dataset('data/alice_in_wonderland.txt', 'data/glove_vocab_10k.txt', window_size=4)
sherlock_data = CBOW.CBOW_Dataset('data/sherlock_holmes_short.txt', 'data/glove_vocab_10k.txt', window_size=4)

In [5]:
alice_model = CBOW.CBOW_Model(nEmbed = 100, vocabSize=alice_data.vocabSize)
alice_trainer = CBOW.CBOW_Trainer(num_epochs = 50, lr = 0.1, batch_size = 32, train_data = alice_data, val_data = alice_data, device = torch.device("cpu"))
alice_trainer.train(alice_model)

Epoch 0:	 Avg Train Loss: 4.78597	 Avg Val Loss: 4.18206
Epoch 20:	 Avg Train Loss: 2.663	 Avg Val Loss: 2.5356
Epoch 40:	 Avg Train Loss: 2.06999	 Avg Val Loss: 1.94868
Training done!
Avg Train Loss: 1.89191


In [6]:
torch.save(alice_model.state_dict(), "alice_cbow_model.pth")

In [7]:
sherlock_model = CBOW.CBOW_Model(nEmbed = 100, vocabSize=sherlock_data.vocabSize)
sherlock_trainer = CBOW.CBOW_Trainer(num_epochs = 50, lr = 0.1, batch_size = 32, train_data = sherlock_data, val_data = sherlock_data, device = torch.device("cpu"))
sherlock_trainer.train(sherlock_model)

Epoch 0:	 Avg Train Loss: 5.09612	 Avg Val Loss: 4.51216
Epoch 20:	 Avg Train Loss: 2.84691	 Avg Val Loss: 2.66648
Epoch 40:	 Avg Train Loss: 2.10947	 Avg Val Loss: 1.97284
Training done!
Avg Train Loss: 1.87833


### Come up with list of words
Answer questions 2 and 3 here

**Same learned embeddings:**

These are common, high-frequency, or function words that appear in similar contexts across both Alice in Wonderland and Sherlock Holmes. Because their meanings and grammatical roles are stable across texts, we expect their embeddings to be similar between the two models.

- talk
- this
- agree
- it
- or
- in
- the
- good
- of
- you

**Different learned embeddings:**

These are content-specific or story-dependent words whose meanings depend heavily on the text’s world and themes.

- magic
- rabbit
- wonder
- queen
- tea
- gentleman
- doctor
- gun
- stock
- tragedy

**Potential Systematic Way to Identify these words**

In [22]:
from collections import Counter
import re

def clean_text(path):
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read().lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    return words

alice_words = clean_text('data/alice_in_wonderland.txt')
sherlock_words = clean_text('data/sherlock_holmes_short.txt')

alice_counts = Counter(alice_words)
sherlock_counts = Counter(sherlock_words)

print(alice_counts.most_common(10))
print(sherlock_counts.most_common(10))

alice_vocab = set(alice_counts.keys())
sherlock_vocab = set(sherlock_counts.keys())

similar_words = [w for w, _ in alice_counts.most_common(200) if w in sherlock_vocab][:10]

print("Similar words:", similar_words)


[('the', 1825), ('and', 913), ('to', 803), ('a', 690), ('of', 631), ('it', 541), ('she', 538), ('said', 462), ('you', 434), ('in', 431)]
[('the', 1327), ('and', 764), ('i', 704), ('to', 672), ('of', 667), ('a', 650), ('in', 426), ('it', 417), ('that', 398), ('you', 378)]
Similar words: ['the', 'and', 'to', 'a', 'of', 'it', 'she', 'said', 'you', 'in']


### Test your hypotheses

Answer question 4 here


In [18]:
def cosine_similarity(vec1, vec2):
    dot_product = torch.dot(vec1, vec2)
    norm1 = torch.norm(vec1)
    norm2 = torch.norm(vec2)
    return dot_product / (norm1 * norm2)

def cosine_similarity_word(word1, word2, embedding_matrix, data):
    idx1 = data.word_to_id[word1]
    idx2 = data.word_to_id[word2]
    vec1 = embedding_matrix[idx1]
    vec2 = embedding_matrix[idx2]
    return cosine_similarity(vec1, vec2).item()

def cosine_similarity_one_word(word, embedding_matrix1, embedding_matrix2, data1, data2):
    idx1 = data1.word_to_id[word]
    idx2 = data2.word_to_id[word]
    vec1 = embedding_matrix1[idx1]
    vec2 = embedding_matrix2[idx2]
    return cosine_similarity(vec1, vec2).item()

In [11]:
alice_embedding_matrix = alice_model.embed.weight.data
sherlock_embedding_matrix = sherlock_model.embed.weight.data
similar = ['talk','this','agree','it','or','in','the','good','of','you']
similar_average_similarity = 0
for word in similar:
    similar_average_similarity += abs(cosine_similarity_one_word(word, alice_embedding_matrix, sherlock_embedding_matrix, alice_data, sherlock_data))
    print(f"Cosine Similarity of '{word}': {cosine_similarity_one_word(word, alice_embedding_matrix, sherlock_embedding_matrix, alice_data, sherlock_data)}")
print(f"Average Cosine Similarity: {similar_average_similarity/len(similar)}")
print("-----")
different = ['magic','rabbit','wonder','queen','tea','gentleman','doctor','gun','stock','tragedy']
varied_average_similarity = 0
for word in different:
    varied_average_similarity += abs(cosine_similarity_one_word(word, alice_embedding_matrix, sherlock_embedding_matrix, alice_data, sherlock_data))
    print(f"Cosine Similarity of '{word}': {cosine_similarity_one_word(word, alice_embedding_matrix, sherlock_embedding_matrix, alice_data, sherlock_data)}")
print(f"Average Cosine Similarity: {varied_average_similarity/len(different)}")

Cosine Similarity of 'talk': 0.07343138009309769
Cosine Similarity of 'this': -0.07191384583711624
Cosine Similarity of 'agree': -0.24359728395938873
Cosine Similarity of 'it': 0.026818489655852318
Cosine Similarity of 'or': -0.19384191930294037
Cosine Similarity of 'in': -0.018676437437534332
Cosine Similarity of 'the': 0.05758483707904816
Cosine Similarity of 'good': 0.18477217853069305
Cosine Similarity of 'of': -0.01208170223981142
Cosine Similarity of 'you': -0.07544316351413727
Average Cosine Similarity: 0.09581612376496196
-----
Cosine Similarity of 'magic': -0.13338904082775116
Cosine Similarity of 'rabbit': -0.0019096520263701677
Cosine Similarity of 'wonder': -0.16419431567192078
Cosine Similarity of 'queen': 0.12570355832576752
Cosine Similarity of 'tea': 0.202909916639328
Cosine Similarity of 'gentleman': -0.02152661792933941
Cosine Similarity of 'doctor': -0.1009579449892044
Cosine Similarity of 'gun': 0.03390151262283325
Cosine Similarity of 'stock': 0.03746004030108452
C

## Part 4 (Optional): Explore the role of other factors on the word embeddings that are learned

#### Embedding Size

In [19]:
embedding_sizes = [25, 100, 300]
related_pairs = [('queen', 'king'), ('man', 'woman'), ('think', 'thought')]
unrelated_pairs = [('queen', 'gun'), ('man', 'tea'), ('think', 'tragedy')]

results = {}

for emb_size in embedding_sizes:
    print(f"Training CBOW model with embedding size = {emb_size}")
    model = CBOW.CBOW_Model(nEmbed=emb_size, vocabSize=alice_data.vocabSize)
    trainer = CBOW.CBOW_Trainer(
        num_epochs=50, lr=0.1, batch_size=32,
        train_data=alice_data, val_data=alice_data,
        device=torch.device("cpu")
    )
    trainer.train(model)
    
    emb = model.embed.weight.data
    related_sims, unrelated_sims = [], []

    for w1, w2 in related_pairs:
        related_sims.append(cosine_similarity_word(w1,w2,emb,alice_data))

    for w1, w2 in unrelated_pairs:
        unrelated_sims.append(cosine_similarity_word(w1,w2,emb,alice_data))
    
    results[emb_size] = {
        'related_avg': sum(related_sims) / len(related_sims),
        'unrelated_avg': sum(unrelated_sims) / len(unrelated_sims)
    }

print("\nEmbedding Size Results:")
for size, res in results.items():
    print(f"Size {size}: Related = {res['related_avg']:.3f}, Unrelated = {res['unrelated_avg']:.3f}")


Training CBOW model with embedding size = 25
Epoch 0:	 Avg Train Loss: 4.9609	 Avg Val Loss: 4.419
Epoch 20:	 Avg Train Loss: 3.43497	 Avg Val Loss: 3.35346
Epoch 40:	 Avg Train Loss: 3.11587	 Avg Val Loss: 3.03231
Training done!
Avg Train Loss: 3.01919
Training CBOW model with embedding size = 100
Epoch 0:	 Avg Train Loss: 4.7765	 Avg Val Loss: 4.17023
Epoch 20:	 Avg Train Loss: 2.67117	 Avg Val Loss: 2.51663
Epoch 40:	 Avg Train Loss: 2.07913	 Avg Val Loss: 1.94505
Training done!
Avg Train Loss: 1.90408
Training CBOW model with embedding size = 300
Epoch 0:	 Avg Train Loss: 4.6432	 Avg Val Loss: 3.88202
Epoch 20:	 Avg Train Loss: 1.9352	 Avg Val Loss: 1.744
Epoch 40:	 Avg Train Loss: 1.43294	 Avg Val Loss: 1.30532
Training done!
Avg Train Loss: 1.31652

Embedding Size Results:
Size 25: Related = 0.251, Unrelated = 0.088
Size 100: Related = 0.200, Unrelated = 0.015
Size 300: Related = 0.076, Unrelated = -0.026


We chose to vary the embedding size, because it directly determines the representational capacity of the model. Smaller embeddings may underfit, while larger embeddings risk overfitting, especially for smaller vocabularies.

Training loss decreased as embedding size increased, showing larger models can fit the data more easily. However, semantic quality peaked at smaller–medium sizes: related pairs remained clearly more similar than unrelated pairs. At 300 dimensions, both related and unrelated similarities collapsed toward zero, implying overfitting or noisy embeddings.

#### Context Size

In [21]:
context_sizes = [2, 4, 6]

results = {}

for ctx_size in context_sizes:
    print(f"Training CBOW model with context window size = {ctx_size}")
    data = CBOW.CBOW_Dataset('data/alice_in_wonderland.txt', 'data/glove_vocab_10k.txt', window_size=ctx_size)
    model = CBOW.CBOW_Model(nEmbed=100, vocabSize=data.vocabSize)
    trainer = CBOW.CBOW_Trainer(
        num_epochs=50, lr=0.1, batch_size=32,
        train_data=data, val_data=data,
        device=torch.device("cpu")
    )
    trainer.train(model)
    
    emb = model.embed.weight.data
    related_sims, unrelated_sims = [], []

    for w1, w2 in related_pairs:
        related_sims.append(cosine_similarity_word(w1,w2,emb,data))

    for w1, w2 in unrelated_pairs:
        unrelated_sims.append(cosine_similarity_word(w1,w2,emb,data))
    
    results[ctx_size] = {
        'related_avg': sum(related_sims) / len(related_sims),
        'unrelated_avg': sum(unrelated_sims) / len(unrelated_sims)
    }

print("\nContext Window Size Results:")
for size, res in results.items():
    print(f"Size {size}: Related = {res['related_avg']:.3f}, Unrelated = {res['unrelated_avg']:.3f}")

Training CBOW model with context window size = 2
Epoch 0:	 Avg Train Loss: 4.66013	 Avg Val Loss: 3.88462
Epoch 20:	 Avg Train Loss: 2.28084	 Avg Val Loss: 2.1104
Epoch 40:	 Avg Train Loss: 1.8322	 Avg Val Loss: 1.69671
Training done!
Avg Train Loss: 1.71997
Training CBOW model with context window size = 4
Epoch 0:	 Avg Train Loss: 4.77016	 Avg Val Loss: 4.17692
Epoch 20:	 Avg Train Loss: 2.66752	 Avg Val Loss: 2.52419
Epoch 40:	 Avg Train Loss: 2.0724	 Avg Val Loss: 1.9346
Training done!
Avg Train Loss: 1.89344
Training CBOW model with context window size = 6
Epoch 0:	 Avg Train Loss: 4.78834	 Avg Val Loss: 4.29431
Epoch 20:	 Avg Train Loss: 2.96986	 Avg Val Loss: 2.83452
Epoch 40:	 Avg Train Loss: 2.34743	 Avg Val Loss: 2.22021
Training done!
Avg Train Loss: 2.14464

Context Window Size Results:
Size 2: Related = 0.210, Unrelated = 0.090
Size 4: Related = 0.103, Unrelated = 0.026
Size 6: Related = -0.027, Unrelated = -0.030


We chose to vary the context window size because it controls how much surrounding context the model uses to learn word representations. Smaller windows focus on very local context, while larger windows incorporate broader context but may introduce noise.

Training loss increased as the window size grew, indicating that larger windows make the prediction task harder. Semantic quality, measured by similarity between related vs. unrelated word pairs, peaked at the smallest window. At size 6, related and unrelated similarities were near zero, suggesting that too large a context window dilutes meaningful relationships.

#### Model Architecture

In [23]:
from CBOW_P4 import CBOW_Model_fc, CBOW_Model_deep

fc_model = CBOW_Model_fc(nEmbed=100, vocabSize=alice_data.vocabSize)
deep_model = CBOW_Model_deep(nEmbed=100, vocabSize=alice_data.vocabSize)

models = {'Fully Connected': fc_model, 'Deep': deep_model}
results = {}

for model_name, model in models.items():
    print(f"Training {model_name} CBOW model")
    trainer = CBOW.CBOW_Trainer(
        num_epochs=50, lr=0.1, batch_size=32,
        train_data=alice_data, val_data=alice_data,
        device=torch.device("cpu")
    )
    trainer.train(model)
    
    emb = model.embed.weight.data
    related_sims, unrelated_sims = [], []

    for w1, w2 in related_pairs:
        related_sims.append(cosine_similarity_word(w1,w2,emb,alice_data))

    for w1, w2 in unrelated_pairs:
        unrelated_sims.append(cosine_similarity_word(w1,w2,emb,alice_data))
    
    results[model_name] = {
        'related_avg': sum(related_sims) / len(related_sims),
        'unrelated_avg': sum(unrelated_sims) / len(unrelated_sims)
    }
    
print("\nModel Architecture Results:")
print ("CBOW Model: Related = 0.200, Unrelated = 0.015")
for model_name, res in results.items():
    print(f"{model_name}: Related = {res['related_avg']:.3f}, Unrelated = {res['unrelated_avg']:.3f}")

Training Fully Connected CBOW model
Epoch 0:	 Avg Train Loss: 4.76747	 Avg Val Loss: 4.18005
Epoch 20:	 Avg Train Loss: 2.65872	 Avg Val Loss: 2.51217
Epoch 40:	 Avg Train Loss: 2.06365	 Avg Val Loss: 1.92759
Training done!
Avg Train Loss: 1.88616
Training Deep CBOW model
Epoch 0:	 Avg Train Loss: 4.66533	 Avg Val Loss: 4.27338
Epoch 20:	 Avg Train Loss: 3.15114	 Avg Val Loss: 2.90674
Epoch 40:	 Avg Train Loss: 2.88954	 Avg Val Loss: 2.60379
Training done!
Avg Train Loss: 2.91877

Model Architecture Results:
CBOW Model: Related = 0.200, Unrelated = 0.015
Fully Connected: Related = 0.159, Unrelated = -0.046
Deep: Related = 0.179, Unrelated = -0.090


We compared the standard CBOW, a fully connected variant, and a deeper CBOW model. All models could fit the training data, but the standard CBOW achieved the best semantic quality. Fully connected and deep variants showed smaller differences between related and unrelated pairs compared to standard CBOW. Some unrelated pairs had slightly negative cosine similarities, indicating that the embeddings sometimes positioned unrelated words in opposite directions.